In [2]:
import pandas as pd

# Load the combined dataset
df = pd.read_csv('all_years_death_data.csv')

# List all cause columns (excluding 'Year' and 'County')
cause_columns = [col for col in df.columns if col not in ['Year', 'County']]

# Dictionary to hold the pivot tables
cause_tables = {}

for cause in cause_columns:
    # Create a pivot table: counties as rows, years as columns, values as death counts
    pivot = df.pivot_table(index='County', columns='Year', values=cause, aggfunc='sum', fill_value=0)
    # Sort columns (years) in order
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)
    cause_tables[cause] = pivot

# Example: Display the table for 'Diseases_of_Heart'
cause_tables['Diseases_of_Heart']

Year,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
County,,,,,,,,,,,,,,,
Adams,199,197,153,199,180,190,171,162,204,193,181,150,156,177,328
Alexander,27,38,28,29,26,22,30,0,32,21,0,20,18,21,36
Bond,43,43,52,35,39,44,36,51,52,48,38,49,67,52,106
Boone,95,80,71,83,84,96,101,0,71,95,0,96,94,113,230
Brown,18,15,14,11,13,13,7,0,10,13,0,12,15,16,24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Wayne,65,57,40,42,45,53,58,0,40,53,0,52,44,51,98
White,237,226,241,204,230,188,193,0,202,260,0,244,213,217,438
Will,189,1047,1081,1162,1076,1125,1153,0,1177,1226,143,142,153,154,2642


In [3]:
import os

# Create a folder to store the output CSVs
output_dir = 'cause_tables'
os.makedirs(output_dir, exist_ok=True)

for cause, table in cause_tables.items():
    # Clean up the cause name for the filename
    safe_cause = cause.replace(' ', '_').replace('/', '_')
    filename = f'{safe_cause}_by_county_year.csv'
    filepath = os.path.join(output_dir, filename)
    table.to_csv(filepath)
    print(f'Saved: {filepath}')

Saved: cause_tables\Total_Deaths_by_county_year.csv
Saved: cause_tables\Diseases_of_Heart_by_county_year.csv
Saved: cause_tables\Malignant_Neoplasms_by_county_year.csv
Saved: cause_tables\Accidents_by_county_year.csv
Saved: cause_tables\COVID_19_by_county_year.csv
Saved: cause_tables\Cerebrovascular_Diseases_by_county_year.csv
Saved: cause_tables\Chronic_Lower_Respiratory_Diseases_by_county_year.csv
Saved: cause_tables\Alzheimers_Disease_by_county_year.csv
Saved: cause_tables\Diabetes_Mellitus_by_county_year.csv
Saved: cause_tables\Nephritis_Nephrotic_Syndrome_Nephrosis_by_county_year.csv
Saved: cause_tables\Influenza_and_Pneumonia_by_county_year.csv
Saved: cause_tables\Septicemia_by_county_year.csv
Saved: cause_tables\Intentional_Self_Harm_by_county_year.csv
Saved: cause_tables\Chronic_Liver_Disease_Cirrhosis_by_county_year.csv
Saved: cause_tables\All_Other_Causes_by_county_year.csv


In [5]:
df = pd.read_csv('all_years_death_data.csv')
df

,Year,County,Total_Deaths,Diseases_of_Heart,Malignant_Neoplasms,Accidents,COVID_19,Cerebrovascular_Diseases,Chronic_Lower_Respiratory_Diseases,Alzheimers_Disease,Diabetes_Mellitus,Nephritis_Nephrotic_Syndrome_Nephrosis,Influenza_and_Pneumonia,Septicemia,Intentional_Self_Harm,Chronic_Liver_Disease_Cirrhosis,All_Other_Causes
0,2008,ILLINOIS,103069,25979,24210,5765,5584,4173,3188,2839,2663,2571,1956,1188,1144,21809,0
1,2008,Adams,843,199,157,49,68,27,22,27,38,26,14,14,6,196,0
2,2008,Alexander,111,27,26,12,2,3,1,7,3,5,4,1,0,20,0
3,2008,Bond,156,43,36,8,9,11,9,2,8,3,1,0,1,25,0
4,2008,Boone,365,95,80,24,26,17,17,13,7,4,7,5,5,65,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1464,2017,White,716,198,132,34,20,59,35,14,11,16,9,0,0,0,0
1465,2017,Will,4776,1073,1094,267,254,257,157,117,130,89,74,0,0,0,0
1466,2017,Will,807,153,166,32,41,61,25,23,23,20,35,0,0,0,0
1467,2017,Winnebago,3056,689,661,145,196,196,159,64,88,45,33,0,0,0,0


In [6]:
import pandas as pd
import numpy as np
import os

# Load the death data
df = pd.read_csv('all_years_death_data.csv')

# Load population data
pop_df = pd.read_csv('../FInal stuff/Illinois Population Data.csv')

# Clean up population data
pop_df = pop_df.rename(columns={'State/City/County': 'County'})

# Convert population columns to numeric, removing commas
for col in pop_df.columns[1:]:
    pop_df[col] = pd.to_numeric(pop_df[col].astype(str).str.replace(',', ''), errors='coerce')

# Create a long format population dataset
pop_long = pop_df.melt(id_vars=['County'], var_name='Year', value_name='Population')
pop_long['Year'] = pop_long['Year'].astype(int)

# Interpolate population for missing years (2008-2022)
years_needed = list(range(2008, 2023))
counties = pop_long['County'].unique()

interpolated_pop = []
for county in counties:
    county_data = pop_long[pop_long['County'] == county].sort_values('Year')
    
    # Create a complete year range
    complete_years = pd.DataFrame({'Year': years_needed})
    county_complete = county_data.merge(complete_years, how='outer', on='Year')
    county_complete['County'] = county
    
    # Interpolate missing population values
    county_complete['Population'] = county_complete['Population'].interpolate(method='linear')
    
    # For years before 1950, use the 1950 value
    county_complete['Population'] = county_complete['Population'].fillna(method='bfill')
    
    # For years after 2020, use the 2020 value
    county_complete['Population'] = county_complete['Population'].fillna(method='ffill')
    
    interpolated_pop.append(county_complete)

pop_interpolated = pd.concat(interpolated_pop, ignore_index=True)

# Merge death data with population data
df_with_pop = df.merge(pop_interpolated, on=['County', 'Year'], how='left')

# Calculate death rates (per 100,000 population)
cause_columns = [col for col in df.columns if col not in ['Year', 'County']]

for cause in cause_columns:
    df_with_pop[f'{cause}_rate'] = (df_with_pop[cause] / df_with_pop['Population']) * 100000

# Create pivot tables for each cause with rates
cause_rate_tables = {}

for cause in cause_columns:
    # Create pivot table with rates
    pivot = df_with_pop.pivot_table(
        index='County', 
        columns='Year', 
        values=f'{cause}_rate', 
        aggfunc='sum', 
        fill_value=0
    )
    
    # Sort columns (years) in order
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)
    
    # Add Illinois total row (average across all counties weighted by population)
    illinois_rates = []
    for year in sorted(pivot.columns):
        year_data = df_with_pop[df_with_pop['Year'] == year]
        if len(year_data) > 0:
            # Calculate weighted average for Illinois
            total_deaths = year_data[cause].sum()
            total_population = year_data['Population'].sum()
            if total_population > 0:
                illinois_rate = (total_deaths / total_population) * 100000
            else:
                illinois_rate = 0
        else:
            illinois_rate = 0
        illinois_rates.append(illinois_rate)
    
    # Add Illinois row
    pivot.loc['ILLINOIS'] = illinois_rates
    
    cause_rate_tables[cause] = pivot

# Display example
print("Example: Diseases of Heart death rates (per 100,000 population)")
cause_rate_tables['Diseases_of_Heart'].head()

C:\Users\eshan\AppData\Local\Temp\ipykernel_14736\4081021718.py:39: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  county_complete['Population'] = county_complete['Population'].fillna(method='bfill')
C:\Users\eshan\AppData\Local\Temp\ipykernel_14736\4081021718.py:42: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  county_complete['Population'] = county_complete['Population'].fillna(method='ffill')
C:\Users\eshan\AppData\Local\Temp\ipykernel_14736\4081021718.py:39: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  county_complete['Population'] = county_complete['Population'].fillna(method='bfill')
C:\Users\eshan\AppData\Local\Temp\ipykernel_14736\4081021718.py:42: FutureWarning: Series.fillna with 'method' is deprecated and will ra

Example: Diseases of Heart death rates (per 100,000 population)


C:\Users\eshan\AppData\Local\Temp\ipykernel_14736\4081021718.py:39: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  county_complete['Population'] = county_complete['Population'].fillna(method='bfill')
C:\Users\eshan\AppData\Local\Temp\ipykernel_14736\4081021718.py:42: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  county_complete['Population'] = county_complete['Population'].fillna(method='ffill')
C:\Users\eshan\AppData\Local\Temp\ipykernel_14736\4081021718.py:39: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  county_complete['Population'] = county_complete['Population'].fillna(method='bfill')
C:\Users\eshan\AppData\Local\Temp\ipykernel_14736\4081021718.py:42: FutureWarning: Series.fillna with 'method' is deprecated and will ra

Year,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
County,,,,,,,,,,,,,,,
Adams,293.139936,291.876355,228.007690,297.163951,269.340923,284.886615,256.924182,243.902439,307.769366,291.775264,274.200048,227.708824,237.309278,269.254758,498.957969
Alexander,295.426362,437.351339,339.888322,365.322113,340.385421,299.784700,426.209013,0.000000,496.956144,342.052969,0.000000,361.023864,343.511450,400.763359,687.022901
Bond,243.240186,242.622581,292.660964,198.146481,222.103261,252.075325,207.483228,295.712173,303.344962,281.724860,224.405915,291.158872,400.597907,310.911809,633.781764
Boone,206.916079,159.876362,131.080956,153.438596,155.493358,177.942869,187.459863,0.000000,132.130389,177.030399,0.000000,179.373203,175.871875,211.420446,430.324802
Brown,259.154389,216.096811,201.816347,160.170071,191.221464,193.190768,105.108262,0.000000,153.346010,201.491034,0.000000,190.074921,240.230621,256.245996,384.368994


In [7]:
# Create a folder to store the output rate CSVs
output_dir = 'death_rate_tables'
os.makedirs(output_dir, exist_ok=True)

for cause, table in cause_rate_tables.items():
    # Clean up the cause name for the filename
    safe_cause = cause.replace(' ', '_').replace('/', '_')
    filename = f'{safe_cause}_death_rates_by_county_year.csv'
    filepath = os.path.join(output_dir, filename)
    table.to_csv(filepath)
    print(f'Saved: {filepath}')

print(f"\nAll {len(cause_rate_tables)} death rate tables have been saved to the '{output_dir}' folder!")

Saved: death_rate_tables\Total_Deaths_death_rates_by_county_year.csv
Saved: death_rate_tables\Diseases_of_Heart_death_rates_by_county_year.csv
Saved: death_rate_tables\Malignant_Neoplasms_death_rates_by_county_year.csv
Saved: death_rate_tables\Accidents_death_rates_by_county_year.csv
Saved: death_rate_tables\COVID_19_death_rates_by_county_year.csv
Saved: death_rate_tables\Cerebrovascular_Diseases_death_rates_by_county_year.csv
Saved: death_rate_tables\Chronic_Lower_Respiratory_Diseases_death_rates_by_county_year.csv
Saved: death_rate_tables\Alzheimers_Disease_death_rates_by_county_year.csv
Saved: death_rate_tables\Diabetes_Mellitus_death_rates_by_county_year.csv
Saved: death_rate_tables\Nephritis_Nephrotic_Syndrome_Nephrosis_death_rates_by_county_year.csv
Saved: death_rate_tables\Influenza_and_Pneumonia_death_rates_by_county_year.csv
Saved: death_rate_tables\Septicemia_death_rates_by_county_year.csv
Saved: death_rate_tables\Intentional_Self_Harm_death_rates_by_county_year.csv
Saved: de

: 